# Exploración y tratamiento de datos

Primer paso del día D (10:00–10:20). Diagnostica la calidad, aplica el tratamiento y explora el objetivo.
Versión sin notebook: `python scripts/explorar_datos.py --datos reto` (genera `outputs/eda/reto/reporte.html`).

> Confidencialidad: mirar agregados; no publicar salidas con filas del reto.

In [ ]:
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
CANDIDATAS = [Path("/content/Hackathon-pnud-riesgo-desplazamiento")]              # repositorio descomprimido en Colab
if EN_COLAB and not (CANDIDATAS[0] / "src" / "hackaton").exists():
    from google.colab import drive                          # si no, la copia del equipo en Drive
    drive.mount("/content/drive")
    CANDIDATAS.append(Path("/content/drive/MyDrive/Documentos Juan Felipe/Hackaton_PENUD"))
CANDIDATAS += [Path.cwd(), Path.cwd().parent]                # ejecución local
RAIZ = next((p for p in CANDIDATAS if (p / "src" / "hackaton").exists()), None)
assert RAIZ is not None, "No encuentro el repositorio: clónalo en /content/Hackathon-pnud-riesgo-desplazamiento o ajusta CANDIDATAS."
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from hackaton import calidad, config, exploracion, graficos
from hackaton.datos import cargar_reto
from hackaton.tratamiento import Transformador, preparar_datos

pd.set_option("display.max_columns", 50)
DATOS = "reto"   # "reto" el día D · "sinteticos_sucios" para practicar con problemas inyectados
crudos = cargar_reto(config.FUENTES_DATOS[DATOS], estricto=False)

## 1. Calidad de los datos crudos
Errores de dominio = datos imposibles (se vuelven nulos y se imputan). Outliers = extremos que pueden ser reales.

In [ ]:
diag = calidad.diagnostico_completo(crudos)
calidad.imprimir_resumen(diag)

In [ ]:
diag["nulos"]

In [ ]:
diag["reglas"].query("violaciones > 0")

In [ ]:
diag["outliers"]

Municipios más extremos de una covariable (cambiar la columna):

In [ ]:
calidad.top_outliers(crudos, "densidad_poblacional", n=10)

## 2. Tratamiento
`preparar_datos` corrige tipos, quita duplicados, corrige el objetivo según la definición oficial e imputa covariables
según su temporalidad (estáticas → mismo municipio; anuales → mismo municipio-año / años vecinos; ONI → mismo mes;
luego mediana del departamento y nacional). Agrega columnas `<col>_imputado`.

In [ ]:
datos, registro = preparar_datos(crudos)
registro["imputacion"]

In [ ]:
diag_tratado = calidad.diagnostico_completo(datos)
calidad.imprimir_resumen(diag_tratado)

## 3. Variable objetivo

In [ ]:
exp = exploracion.exploracion_completa(datos["entrenamiento"])
exploracion.imprimir_resumen(exp)
figs = graficos.figuras_exploracion(exp, datos)
figs["serie_nacional"].show()

In [ ]:
figs["estacionalidad"].show()
figs["fase_enso"].show()
figs["magnitud"].show()

In [ ]:
exp["tasa_por_anio"]

## 4. Geografía del riesgo

In [ ]:
figs["departamentos"].show()
figs["tasas_municipales"].show()

## 5. Covariables y cambio de distribución
Ojo con `pct_fuera_rango`: los árboles no extrapolan (el ONI de El Niño 2023 supera el máximo del entrenamiento).

In [ ]:
figs["covariables"].show()
figs["auc_univariado"].show()
diag_tratado["cambio_distribucion"]

## 6. Línea base de climatología (piso a superar)

In [ ]:
exp["linea_base_climatologia"]

## 7. Transformaciones para modelar
`Transformador` aplica log1p a las asimétricas, recorta por percentiles y (opcional) escala. Se ajusta SOLO con la
historia del fold. Los árboles no necesitan escalado; los modelos lineales sí (`escalar="estandar"` o `"robusto"`).

In [ ]:
historia = datos["entrenamiento"]
t = Transformador(escalar="robusto").fit(historia)
t.transform(datos["prueba_oculta"])[[c for c in t.transform(historia.head()).columns if c.startswith("log_")]].describe()

## 8. Decisiones del equipo (completar)

| Tema | Hallazgo | Decisión |
|---|---|---|
| Nulos | | |
| Duplicados | | |
| Outliers en covariables | | |
| Outliers en personas (objetivo) | | |
| Escalado | | |
| Covariables con cambio de distribución | | |

Registrar las decisiones importantes también en `.claude/skills/hackaton-pnud-desplazamiento/references/estrategia.md`.